In [2]:
import pandas as pd 
import numpy as np 

In [3]:
import h5py

filepath = "inputs/wind-ons-reference_county.h5"

with h5py.File(filepath, "r") as f:
    cf = f["cf"][:]          # NumPy array
    index = f["index"][:]    # time labels
    columns = f["columns"][:]  # county labels


In [4]:
index = index.astype(str)
columns = columns.astype(str)
df = pd.DataFrame(cf, index=index, columns=columns)

print(df.head())
print(df.shape)
print(df.index[:5])



   3_p01001  4_p01001  5_p01001  4_p01003  5_p01003  4_p01005  5_p01005  \
1  0.289062  0.584473  0.722656  0.973145  0.979004  0.194458  0.258057   
2  0.283936  0.455322  0.678223  0.991211  0.964355  0.273926  0.293945   
3  0.406006  0.432373  0.570801  0.897949  0.916992  0.215332  0.293213   
4  0.451904  0.496338  0.688477  0.903809  0.890625  0.362549  0.280518   
5  0.358887  0.467285  0.648438  0.846191  0.833984  0.255371  0.214355   

   4_p01007  5_p01007  3_p01009  ...  4_p56043  5_p56043  7_p56043  10_p56043  \
1  0.832520  0.860840  0.329102  ...       0.0       0.0  0.031006   0.111023   
2  0.734863  0.748047  0.263916  ...       0.0       0.0  0.049011   0.106995   
3  0.688965  0.650391  0.192993  ...       0.0       0.0  0.068970   0.062012   
4  0.750488  0.680664  0.133057  ...       0.0       0.0  0.075012   0.049988   
5  0.668945  0.675293  0.159058  ...       0.0       0.0  0.061005   0.044006   

   4_p56045  5_p56045  6_p56045  7_p56045  8_p56045  9_p56045 

C:\Users\henry\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pandas\io\formats\format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


In [5]:
cols = df.columns.to_series()

meta = cols.str.extract(
    r"(?P<class>\d+)_p(?P<fips>\d+)"
)

meta["class"] = meta["class"].astype(int)
meta["fips"] = meta["fips"].astype(str).str.zfill(5)


In [6]:
df.columns = pd.MultiIndex.from_frame(meta)
df.columns.names = ["class", "fips"]


In [7]:
county_cf = df.groupby(level="fips", axis=1).mean()


C:\Users\henry\AppData\Local\Temp\ipykernel_24484\3585328010.py:1: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  county_cf = df.groupby(level="fips", axis=1).mean()


In [8]:
# Get FIPS translator 

fips_translate_filename = "inputs/state_and_county_fips_master.csv"

fips_translator = pd.read_csv(fips_translate_filename, dtype={"fips": str})
fips_translator['fips'] = fips_translator['fips'].apply(lambda x: str(x).zfill(5))

In [9]:
# Step 1: Create a mapping from fips number to "State_County"
fips_map = fips_translator.set_index('fips').apply(lambda row: f"{row['state']}_{row['name']}", axis=1).to_dict()

new_columns = pd.MultiIndex.from_tuples(
    [(cls, fips_map.get(fips, fips)) for cls, fips in df.columns],
    names=df.columns.names
)

df.columns = new_columns



In [10]:
print(df.head())

class                3                 4                 5                 4   \
fips  AL_Autauga County AL_Autauga County AL_Autauga County AL_Baldwin County   
1              0.289062          0.584473          0.722656          0.973145   
2              0.283936          0.455322          0.678223          0.991211   
3              0.406006          0.432373          0.570801          0.897949   
4              0.451904          0.496338          0.688477          0.903809   
5              0.358887          0.467285          0.648438          0.846191   

class                5                 4                 5              4   \
fips  AL_Baldwin County AL_Barbour County AL_Barbour County AL_Bibb County   
1              0.979004          0.194458          0.258057       0.832520   
2              0.964355          0.273926          0.293945       0.734863   
3              0.916992          0.215332          0.293213       0.688965   
4              0.890625          0.362549 

C:\Users\henry\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pandas\io\formats\format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


In [11]:
PJM_states = ['PA', 'NJ', 'MD', 'DC', 'DE', 'IL', 'WV', 'VA', 'OH', 'IN', 'MI', 'KY', 'NC']  # added MI, KY, NC as commonly in PJM


# 2️⃣ Boolean mask for columns where the state is in PJM_states
mask = df.columns.get_level_values(1).str[:2].isin(PJM_states)

# 3️⃣ Filter columns
PJM_df = df.loc[:, mask]


In [12]:
PJM_frame = PJM_df.mean(axis=1).to_frame(name='pjm_avg')
PJM_frame.index = PJM_frame.index.astype(int)

PJM_frame['day'] = PJM_frame.index // 24


In [13]:
weather = pd.read_csv("inputs/solar_weather_data.csv")
thi_series = weather['system_max_thi']

PJM_frame['thi'] = thi_series

PJM_frame['thi'] = thi_series

print(PJM_frame)


        pjm_avg   day        thi
1      0.848633     0  57.432200
2      0.836914     0  56.804000
3      0.830566     0  56.174000
4      0.812500     0  57.223400
5      0.789551     0  58.605589
...         ...   ...        ...
61316  0.834961  2554  62.255973
61317  0.844727  2554  62.817728
61318  0.850586  2554  60.487492
61319  0.855957  2554  56.683400
61320  0.858887  2555  53.083397

[61320 rows x 3 columns]


C:\Users\henry\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pandas\io\formats\format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


In [14]:
wind_performance_dataframe = PJM_frame.copy()
print(wind_performance_dataframe.head(24))

     pjm_avg  day        thi
1   0.848633    0  57.432200
2   0.836914    0  56.804000
3   0.830566    0  56.174000
4   0.812500    0  57.223400
5   0.789551    0  58.605589
6   0.765625    0  59.265292
7   0.731445    0  59.768344
8   0.697266    0  60.026889
9   0.626953    0  60.675020
10  0.536621    0  61.451061
11  0.514160    0  61.833970
12  0.531738    0  62.037908
13  0.572266    0  62.124884
14  0.613281    0  62.430400
15  0.637207    0  63.009694
16  0.645996    0  63.337932
17  0.739258    0  64.141077
18  0.825195    0  64.600890
19  0.859375    0  65.096168
20  0.872559    0  65.290886
21  0.861816    0  64.759673
22  0.823242    0  64.064048
23  0.770508    0  63.929366
24  0.705078    1  63.929366


C:\Users\henry\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pandas\io\formats\format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


In [15]:
wind_performance_dataframe['datetime'] = weather['date']
cols_to_keep = ['datetime', 'pjm_avg', 'thi']
wind_performance_dataframe = wind_performance_dataframe[cols_to_keep]

wind_performance_dataframe['datetime'] = pd.to_datetime(
    wind_performance_dataframe['datetime']
)

wind_performance_dataframe['season'] = np.where(
    wind_performance_dataframe['datetime'].dt.month.isin([10, 11, 12, 1, 2, 3]),
    'winter', 
    'summer'
)

wind_performance_dataframe['date'] = wind_performance_dataframe['datetime'].dt.date

In [16]:
daily_thi = (
    wind_performance_dataframe.groupby(["date", "season"])["thi"]
      .agg(
          thi_extreme=lambda x: x.max() if x.name[1] == "summer" else x.min()
      )
      .reset_index()
)

bin_edges = np.arange(daily_thi["thi_extreme"].min() // 5 * 5,
                      daily_thi["thi_extreme"].max() + 5,
                      5)

daily_thi["thi_bin"] = pd.cut(
    daily_thi["thi_extreme"],
    bins=bin_edges
)


In [22]:
daily_profiles = (
    wind_performance_dataframe.sort_values("datetime")
      .groupby("date")["pjm_avg"]
      .apply(lambda x: x.values if len(x) == 24 else None)
      .reset_index(name="wind_profile")
)
daily_profiles = daily_profiles.dropna()


In [23]:
daily = daily_thi.merge(daily_profiles, on="date", how="inner")
daily.head()

,date,season,thi_extreme,thi_bin,wind_profile
0,2007-01-02,winter,44.173400,"(40.0, 45.0]","[0.705, 0.6367, 0.5737, 0.5244, 0.4858, 0.453,..."
1,2007-01-03,winter,37.693400,"(35.0, 40.0]","[0.3857, 0.401, 0.4224, 0.4458, 0.474, 0.5, 0...."
2,2007-01-04,winter,49.753400,"(45.0, 50.0]","[0.76, 0.7607, 0.7603, 0.758, 0.757, 0.7515, 0..."
3,2007-01-05,winter,58.115310,"(55.0, 60.0]","[0.8066, 0.7935, 0.781, 0.764, 0.745, 0.713, 0..."
4,2007-01-06,winter,63.765006,"(60.0, 65.0]","[0.68, 0.6914, 0.702, 0.712, 0.713, 0.709, 0.7..."


In [24]:
long = (
    daily
    .explode("wind_profile")
    .assign(hour=lambda x: x.groupby("date").cumcount())
)

long.to_csv("daily_onshore_profiles_long.csv", index=False)

print(long)

            date  season  thi_extreme       thi_bin wind_profile  hour
0     2007-01-02  winter      44.1734  (40.0, 45.0]     0.705078     0
0     2007-01-02  winter      44.1734  (40.0, 45.0]     0.636719     1
0     2007-01-02  winter      44.1734  (40.0, 45.0]      0.57373     2
0     2007-01-02  winter      44.1734  (40.0, 45.0]     0.524414     3
0     2007-01-02  winter      44.1734  (40.0, 45.0]      0.48584     4
...          ...     ...          ...           ...          ...   ...
2553  2013-12-29  winter      42.0134  (40.0, 45.0]      0.82959    19
2553  2013-12-29  winter      42.0134  (40.0, 45.0]     0.834961    20
2553  2013-12-29  winter      42.0134  (40.0, 45.0]     0.844727    21
2553  2013-12-29  winter      42.0134  (40.0, 45.0]     0.850586    22
2553  2013-12-29  winter      42.0134  (40.0, 45.0]     0.855957    23

[61296 rows x 6 columns]
